# Week 7 — Wednesday: Choosing a Model — Cross-Validation and Grid Search

**DATA 202 · Calvin University**

Same children, same two blood measurements (`fGSH`, `GSSG`), same split: 111 training children, 48 test children.

Monday, we chose depth 2 by looking at the **test** children's scores.

> 🎯 **How can we choose a tree's depth without touching the test set?**

**Today's plan (~40 min):**

| Time | Part |
|---|---|
| ~7 min | 1 · One test set can't choose |
| ~10 min | 2 · Cross-validation: every child takes a turn (SLO 07B) |
| ~8 min | 3 · Choosing depth with cross-validation (SLO 07B) |
| ~10 min | 4 · Grid search: try every combination (SLO 07C) |
| ~5 min | 5 · The best of many |

**Cues:** 🎯 Predict First · 💬 Question · 🗣️ Pair Talk · 📌 One split

In [ ]:
import pandas as pd
import plotly.express as px
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

autism = pd.read_csv("https://cs.calvin.edu/courses/data/202/26fa/datasets/autism.csv")
kids = autism[autism["Group"] != "SIB"]                    # ASD vs. NEU
X = kids[["fGSH", "GSSG"]]
y = kids["Group"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=7)   # Monday's split

---
## 1 · One Test Set Can't Choose · ~7 min

> 🎯 **Predict First:** Monday's split said depth 2 or 3 is best. With a different random split, would the best depth be the same?

In [ ]:
rows = []
for seed in [7, 1, 13, 21, 30]:                           # Monday's split first, then four others
    X_a, X_b, y_a, y_b = train_test_split(X, y, test_size=0.3, random_state=seed)
    for depth in range(1, 6):
        model = DecisionTreeClassifier(max_depth=depth, random_state=0).fit(X_a, y_a)
        rows.append({"random_state": seed, "max_depth": depth, "test accuracy": accuracy_score(y_b, model.predict(X_b))})

pd.DataFrame(rows).pivot(index="random_state", columns="max_depth", values="test accuracy").round(3)

> 📄 **Handout 1 — Five splits.** In each row, circle the best depth. Write down the lowest and highest score for depth 2. 🗣️ Which depth would you pick?

<details><summary>Answer</summary>

* Best depth: **2 or 3** in four splits, **4** in one
* Depth 2 scores from **77.1%** to **93.8%** — same tree recipe, same children
* Two problems with choosing on the test set:
  1. **Luck:** one split of 48 children is a noisy judge
  2. **Peeking:** if the test set picks the depth, it's no longer "new data", and its score is no longer honest
</details>

**The fix:** leave the test set **locked** until the very end. Do the choosing *inside* the training set, and do it **many times**.

---
## 2 · Cross-Validation: Every Child Takes a Turn (SLO 07B) · ~10 min

<img src="https://cs.calvin.edu/courses/data/202/26fa/weeks/07/images/cv_folds.png" width="720" alt="Diagram of 5-fold cross-validation. The 111 training children are cut into 5 folds of about 22. In each of 5 rounds, one fold (orange, 'validate') is held out and the other four (blue, 'fit') train the tree; the rounds score 0.870, 0.818, 0.864, 0.909 and 0.773. The cross-validation score is their mean, 0.847. The 48 test children sit in a gray 'locked' box in every round.">

**5-fold cross-validation:**

1. Cut the **training** children into 5 groups: the **folds**
2. Round 1: fit on folds 2–5, score on fold 1. Round 2: fit on 1, 3, 4, 5, score on fold 2. And so on
3. Every child is scored **exactly once**, by a tree that never saw them
4. The **mean** of the 5 scores is the cross-validation score

In [ ]:
from sklearn.model_selection import cross_val_score

fold_scores = cross_val_score(DecisionTreeClassifier(max_depth=2, random_state=0), X_train, y_train, cv=5)
print("5 rounds:", fold_scores.round(3))
print("mean:    ", fold_scores.mean().round(3))

> 📄 **Handout 2 — Five folds.** In each round, shade the fold that is scored. Then compute the mean of the 5 scores.

<details><summary>Answer</summary>

* (0.870 + 0.818 + 0.864 + 0.909 + 0.773) ÷ 5 = **0.847**
* From 77.3% to 90.9% between rounds: the same luck we saw in Part 1, now **averaged out**
* `cross_val_score` fit **5 trees**. The 48 test children were never touched
</details>

---
## 3 · Choosing Depth With Cross-Validation (SLO 07B) · ~8 min

One cross-validation score per depth. The error bars show how much the 5 rounds disagreed.

In [ ]:
cv_scores = []
for depth in range(1, 9):
    s = cross_val_score(DecisionTreeClassifier(max_depth=depth, random_state=0), X_train, y_train, cv=5)
    cv_scores.append({"max_depth": depth, "mean accuracy": s.mean(), "spread": s.std()})
cv_scores = pd.DataFrame(cv_scores)

fig = px.line(cv_scores, x="max_depth", y="mean accuracy", error_y="spread", markers=True,
              title="5-Fold Cross-Validation Accuracy vs. Depth (111 training children)")
fig.show()

> 📄 **Handout 3 — Pick the depth.** Circle the depth with the best cross-validation score. Label the underfitting and overfitting sides.

<details><summary>Answer</summary>

* **Best: depth 2** (0.847)
* **Depth 1:** 0.702 → **underfitting**
* **Depth 7 and up:** 0.784 → **overfitting**: the full tree memorizes the 4 folds it fits on
* Depths 2–6 are within about 3 points of each other — and the error bars overlap
</details>

Now, **once**, open the locked test set:

In [ ]:
final_tree = DecisionTreeClassifier(max_depth=2, random_state=0).fit(X_train, y_train)   # all 111 training children
print("test accuracy:", round(accuracy_score(y_test, final_tree.predict(X_test)), 3))

* Same 87.5% as Monday, but now it's an honest score: the test set chose nothing

> 📌 **One split:** 87.5% is on *these* 48 test children.

---
## 4 · Grid Search: Try Every Combination (SLO 07C) · ~10 min

A tree has more than one knob. Monday's pruning knobs: `max_depth` and `min_samples_leaf`.

**Grid search:** list values for each knob, cross-validate **every combination**, keep the best.

In [ ]:
from sklearn.model_selection import GridSearchCV

grid = {"max_depth": [1, 2, 3, 4, 5, 6, None],           # None = no limit
        "min_samples_leaf": [1, 2, 5, 10, 20]}

search = GridSearchCV(DecisionTreeClassifier(random_state=0), grid, cv=5)
search.fit(X_train, y_train)                             # training children only

In [ ]:
results = pd.DataFrame(search.cv_results_)
results["max_depth"] = [str(p["max_depth"]).replace("None", "no limit") for p in results["params"]]
table = results.pivot(index="param_min_samples_leaf", columns="max_depth", values="mean_test_score")
table = table[["1", "2", "3", "4", "5", "6", "no limit"]]

fig = px.imshow(table, text_auto=".3f", color_continuous_scale="Blues", aspect="auto",
                labels=dict(x="max_depth", y="min_samples_leaf", color="CV accuracy"),
                title="Grid Search: 5-Fold CV Accuracy for 35 Combinations")
fig.show()

> 📄 **Handout 4 — The grid.** Circle the best cell. How many trees did the grid search fit? Box every cell within 0.01 of the best.

In [ ]:
print("best combination:", search.best_params_)
print("best CV accuracy:", round(search.best_score_, 3))

<details><summary>Answer</summary>

* **Best:** `max_depth=6`, `min_samples_leaf=5` → **0.856**. "No limit" with the same leaf rule ties exactly: it's the same tree, which stops at depth 4 anyway
* **Trees fitted:** 35 combinations × 5 folds = **175** (plus one final refit of the winner on all 111 children)
* Depth 3 also shows 0.856 (a hair less), and Monday's depth 2 scores 0.847: **1 child** out of 111 behind the winner
</details>

> 🎯 **Predict First:** On the locked test set, which does better — the grid's winner, or plain depth 2?

In [ ]:
winner = search.best_estimator_                          # already refit on all 111 training children
print("grid winner | CV:", round(search.best_score_, 3), "| test:", round(accuracy_score(y_test, winner.predict(X_test)), 3))
print("depth 2     | CV:", round(cv_scores.loc[1, "mean accuracy"], 3), "| test:", round(accuracy_score(y_test, final_tree.predict(X_test)), 3))

<details><summary>Answer</summary>

* Winner: 85.6% in cross-validation, **79.2%** on test. Depth 2: 84.7% → **87.5%**
* The winner was the **best of 35**, and part of being best was luck in these 5 folds. The best of many scores is **optimistic**
* When scores are within a child or two, prefer the **simpler** model
* The locked test set is the honest check, and it is still **one split** of 48 children 📌
</details>

---
## 5 · The Best of Many · ~5 min

The 2017 study behind our data used a different model, and searched over **which metabolites** to use:

> "To avoid over-fitting and over-stating results, leave-one-out cross validation is employed …"
>
> "Therefore, all combinations of up to six variables were evaluated for separability."
> — Howsmon et al. (2017)

* *Leave-one-out*: cross-validation with one fold per child
* With 24 measurements, "all combinations of up to six" is about **190,000** combinations, each cross-validated. The paper reports the **maximum** score for each number of metabolites

> 💬 **Question:** Every one of those scores was cross-validated. Is the *best* of them a fair estimate for the next child who walks into a clinic? What would be?

<details><summary>Answer</summary>

* Our grid: 35 tries, and the winner dropped from 85.6% to 79.2% on fresh children. With ~190,000 tries, the best has many more chances to be lucky
* Cross-validation makes **each** score fair. **Choosing the maximum** is one more step, and it needs its own untouched data
* A fair check: children used for **no** choice at all, ideally from another clinic. All 159 children came from one study in Arkansas
* This doesn't make the paper wrong. It means its best number needs a **new test**
</details>

---
### Wrap-up

1. **Test set:** locked until the end, opened **once**
2. **Cross-validation:** split the training set into folds; every child is scored once by a model that never saw it
3. **Tuning:** pick each knob by its cross-validation score; **grid search** tries every combination
4. **The best of many is optimistic:** when scores are close, prefer the simpler model 📌

**Practice 06 (after Fall Break):** trees, cross-validation and grid search on your own.